# Fiabilité des scores des CNN

**Objectif :** examiner si les scores PNEUMONIA correspondent aux fréquences observées dans les étiquettes du dataset. Par exemple, parmi des images ayant un score proche de 0,90, une proportion observée proche de 90 % est compatible avec une bonne calibration.

Ce notebook exploite les prédictions sauvegardées par les notebooks **08 et 10**. Il compare les deux architectures, les trois graines (42, 123, 2026), les sept conditions d’image et les deux variantes d’apprentissage. Cela représente **84 fichiers de prédictions**, concernant toujours les mêmes **953 images de validation**.

**Exécution :** enregistrer ce notebook dans le dossier du projet, choisir le noyau **memoire_cnn**, puis exécuter les cellules dans l’ordre. Les modèles et les radiographies ne sont pas rechargés. Aucun entraînement ni ajustement de seuil ou de calibration n’est effectué.

Les sorties sont enregistrées dans **optimisation/fiabilite_scores_v1**. Les fichiers des expériences restent les sources de cette analyse.

Les modèles et leurs meilleures époques ont déjà été sélectionnés sur cette validation. Le test avait aussi été consulté avant l’optimisation. Les résultats restent donc **exploratoires** : ils décrivent les scores sur ces données et ne mesurent pas leur fiabilité dans un nouvel établissement ou chez de nouveaux patients.

In [ ]:
from pathlib import Path
import hashlib
import itertools
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
SOURCES = {
    "sans": PROJET / "optimisation" / "robustesse_validation_v1",
    "avec": PROJET / "optimisation" / "robustesse_augmentation_v1",
}
COMPARAISON = SOURCES["avec"] / "comparaison_robustesse_augmentation.csv"
SORTIE = PROJET / "optimisation" / "fiabilite_scores_v1"
MODELES = ["CNN_classique", "CNN_CBAM"]
GRAINES = [42, 123, 2026]
CONDITIONS = [
    "originale", "luminosite_090", "luminosite_110",
    "contraste_090", "contraste_110", "bruit_001", "bruit_003",
]
LIBELLES = {
    "originale": "Images originales",
    "luminosite_090": "Luminosité × 0,90",
    "luminosite_110": "Luminosité × 1,10",
    "contraste_090": "Contraste × 0,90",
    "contraste_110": "Contraste × 1,10",
    "bruit_001": "Bruit σ = 0,01",
    "bruit_003": "Bruit σ = 0,03",
}
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
CLES = ["modele", "graine", "condition"]
COUPLES = set(itertools.product(MODELES, GRAINES))
COMBINAISONS = set(itertools.product(MODELES, GRAINES, CONDITIONS))
N_INTERVALLES = 10
BORNES = np.linspace(0.0, 1.0, N_INTERVALLES + 1)
# Seuil descriptif des erreurs à score élevé, sans modifier les décisions.
CONFIANCE_EXAMINEE = 0.90

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def id_protocole(protocole):
    texte = json.dumps(protocole, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(texte.encode("utf-8")).hexdigest()

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(
        json.dumps(objet, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporaire.replace(chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(
        temporaire, index=False, encoding="utf-8-sig",
        float_format="%.17g",
    )
    temporaire.replace(chemin)

print("Analyse des scores sauvegardés : 12 modèles, 7 conditions.")

## 1. Retrouver les fichiers et vérifier leur provenance

Les listes d’images, les groupes, les étiquettes, les graines et les époques doivent correspondre aux expériences précédentes. Les empreintes enregistrées servent à détecter un mélange de fichiers.

En cas de fichier de prédictions absent, le message indique son emplacement. Les notebooks 08 et 10 permettent de recréer ces prédictions à partir des modèles sauvegardés.

In [ ]:
fichier_val = REPARTITION / "val.csv"
requis = [
    fichier_val, COMPARAISON,
    SOURCES["sans"] / "resultats_robustesse.csv",
    *[d / "protocole.json" for d in SOURCES.values()],
]
absents = [str(p) for p in requis if not p.is_file()]
exiger(not absents, "Fichiers nécessaires introuvables :\n" + "\n".join(absents))
val_df = pd.read_csv(
    fichier_val, dtype={"groupe": str, "chemin_relatif": str}
)
exiger(
    {"chemin_relatif", "groupe", "classe"}.issubset(val_df.columns),
    "Le fichier val.csv ne contient pas les colonnes attendues.",
)
exiger(
    len(val_df) == 953 and val_df["chemin_relatif"].is_unique,
    "Effectif ou identifiants de validation incompatibles.",
)
exiger(
    not val_df[["chemin_relatif", "groupe", "classe"]].isna().any().any(),
    "Une image de validation manque d’identifiant, de groupe ou de classe.",
)
exiger(
    val_df["classe"].value_counts().to_dict() == {"PNEUMONIA": 684, "NORMAL": 269},
    "Effectifs par classe incompatibles.",
)
IDENTIFIANTS = val_df["chemin_relatif"].tolist()
GROUPES = val_df["groupe"].tolist()
REELS = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
SHA_VAL = empreinte_fichier(fichier_val)
PROTOCOLES, INFORMATIONS, IDS = {}, {}, {}
for variante, dossier in SOURCES.items():
    protocole = json.loads((dossier / "protocole.json").read_text(encoding="utf-8"))
    version = (
        "robustesse_validation_v1" if variante == "sans"
        else "comparaison_robustesse_augmentation_v1"
    )
    exiger(protocole["version_analyse"] == version, "Autre version d’analyse.")
    exiger(
        protocole["ensemble"] == "val"
        and protocole["effectif"] == len(val_df)
        and protocole["sha256_val_csv"] == SHA_VAL
        and protocole["classes"] == CLASSES,
        "Protocole d’un autre ensemble ou d’une autre correspondance de classes.",
    )
    exiger(
        [c["id"] for c in protocole["conditions"]] == CONDITIONS,
        "Conditions d’évaluation différentes.",
    )
    informations = {
        (m["modele"], int(m["graine"])): m for m in protocole["modeles"]
    }
    exiger(
        len(protocole["modeles"]) == 6 and set(informations) == COUPLES,
        "Il faut les six couples modèle/graine attendus dans chaque protocole.",
    )
    PROTOCOLES[variante] = protocole
    INFORMATIONS[variante] = informations
    IDS[variante] = id_protocole(protocole)

for cle in [
    "conditions", "lot", "appareil", "classes", "pretraitement",
    "sha256_val_csv", "sha256_images_preparees_et_masques",
    "bruit", "zone_perturbee", "contraste", "ecretage", "prediction",
]:
    exiger(
        PROTOCOLES["sans"][cle] == PROTOCOLES["avec"][cle],
        f"Évaluations non comparables : {cle}.",
    )
exiger(
    PROTOCOLES["avec"]["sha256_protocole_sans_augmentation"]
    == empreinte_fichier(SOURCES["sans"] / "protocole.json"),
    "Le protocole de référence a changé depuis la comparaison.",
)
exiger(
    PROTOCOLES["avec"]["sha256_bilan_sans_augmentation"]
    == empreinte_fichier(SOURCES["sans"] / "resultats_robustesse.csv"),
    "Le bilan de référence a changé depuis la comparaison.",
)
for couple in COUPLES:
    exiger(
        INFORMATIONS["avec"][couple]["sha256_reference_sans_augmentation"]
        == INFORMATIONS["sans"][couple]["sha256_sauvegarde"],
        "Association incorrecte entre les modèles avec et sans augmentation.",
    )

bilan = pd.read_csv(COMPARAISON, float_precision="round_trip")
exiger(set(CLES).issubset(bilan.columns), "Comparaison incomplète.")
exiger(
    len(bilan) == 42 and not bilan.duplicated(CLES).any()
    and set(map(tuple, bilan[CLES].values)) == COMBINAISONS,
    "La comparaison doit contenir les 42 couples de résultats attendus.",
)
BILAN = bilan.set_index(CLES)
print("Validation et protocoles vérifiés :", len(val_df), "images.")

## 2. Définir les mesures de fiabilité des scores

| Mesure | Ce qu’elle décrit | Lecture |
|---|---|---|
| Brier binaire | Moyenne de \((p-y)^2\), avec \(p\) le score PNEUMONIA et \(y\) l’étiquette 0 ou 1 | Plus petit = meilleure qualité globale des scores |
| Perte logarithmique | Pénalise particulièrement une erreur accompagnée d’un score très affirmé | Plus petite = meilleure qualité globale des scores |
| Écart de calibration PNEUMONIA | Moyenne pondérée des écarts absolus entre score moyen et fréquence observée, dans 10 intervalles fixes | Plus petit = meilleur accord pour ce découpage |
| Erreurs avec confiance ≥ 0,90 | Erreurs dont le score de la classe prédite atteint 0,90 | Lire le nombre d’erreurs **et** le nombre de prédictions concernées |

Les calculs donnent **le même poids à chaque image**. La perte logarithmique ici n’est donc pas la perte pondérée utilisée pendant l’entraînement. Les résultats de Brier et de perte logarithmique sont aussi détaillés selon la vraie classe.

Le Brier et la perte logarithmique mesurent plusieurs propriétés des scores : une baisse ne prouve pas, à elle seule, une meilleure calibration. Les courbes et leurs effectifs complètent leur lecture. Les intervalles vides restent vides.

Références : [calibration et courbes de fiabilité, scikit-learn](https://scikit-learn.org/stable/modules/calibration.html) ; [définition du score de Brier](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html).

In [ ]:
def probabilites_pneumonie(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    exponentielles = np.exp(z)
    return exponentielles[:, 1] / exponentielles.sum(axis=1)

def verifier_predictions(tableau, variante, nom, graine, condition):
    necessaires = {
        "chemin_relatif", "groupe", "classe_vraie", "prediction",
        "logit_NORMAL", "logit_PNEUMONIA", "score_pneumonie",
        "modele", "graine", "epoque", "condition", "id_protocole",
    }
    exiger(necessaires.issubset(tableau.columns), "Prédictions incomplètes.")
    exiger(
        len(tableau) == len(val_df)
        and tableau["chemin_relatif"].tolist() == IDENTIFIANTS,
        "Ordre ou identifiants d’images différents.",
    )
    exiger(tableau["groupe"].tolist() == GROUPES, "Groupes différents.")
    exiger(
        np.array_equal(tableau["classe_vraie"].to_numpy(), REELS),
        "Étiquettes différentes.",
    )
    information = INFORMATIONS[variante][(nom, graine)]
    attendus = {
        "modele": nom, "graine": graine, "epoque": information["epoque"],
        "condition": condition, "id_protocole": IDS[variante],
    }
    for cle, valeur in attendus.items():
        exiger((tableau[cle] == valeur).all(), f"Métadonnée incorrecte : {cle}.")
    logits = tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(np.float64)
    p = tableau["score_pneumonie"].to_numpy(np.float64)
    exiger(np.isfinite(logits).all(), "Scores bruts non finis.")
    exiger(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), "Scores invalides.")
    exiger(
        np.allclose(p, probabilites_pneumonie(logits), atol=1e-10, rtol=0),
        "Scores PNEUMONIA incompatibles avec les logits.",
    )
    exiger(
        np.array_equal(tableau["prediction"].to_numpy(), logits.argmax(axis=1)),
        "Prédictions incompatibles avec les logits.",
    )
    ligne = BILAN.loc[(nom, graine, condition)]
    exiger(
        ligne[f"epoque_{variante}"] == information["epoque"]
        and ligne[f"n_images_{variante}"] == len(tableau),
        "La comparaison et le fichier de prédictions ne correspondent pas.",
    )

def courbe_calibration(y, p):
    # [0 ; 0,1[, [0,1 ; 0,2[, ... [0,9 ; 1].
    indices = np.clip(
        np.searchsorted(BORNES, p, side="right") - 1, 0, N_INTERVALLES - 1
    )
    lignes = []
    for i in range(N_INTERVALLES):
        masque = indices == i
        n = int(masque.sum())
        lignes.append({
            "intervalle": i + 1,
            "borne_inferieure": float(BORNES[i]),
            "borne_superieure": float(BORNES[i + 1]),
            "n_images": n,
            "score_moyen": float(p[masque].mean()) if n else np.nan,
            "fraction_PNEUMONIA": float(y[masque].mean()) if n else np.nan,
        })
    return pd.DataFrame(lignes)

def calculer_fiabilite(tableau):
    y = tableau["classe_vraie"].to_numpy(dtype=np.int64)
    logits = tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(np.float64)
    p = probabilites_pneumonie(logits)
    prediction = logits.argmax(axis=1)
    erreur = prediction != y
    brier_image = (p - y) ** 2
    # Calcul stable même pour un score très proche de 0 ou de 1.
    marge = logits[:, 1] - logits[:, 0]
    perte_image = np.logaddexp(0.0, (1 - 2 * y) * marge)
    courbe = courbe_calibration(y, p)
    occupees = courbe[courbe["n_images"] > 0]
    ece = np.sum(
        (occupees["n_images"] / len(y))
        * abs(occupees["fraction_PNEUMONIA"] - occupees["score_moyen"])
    )
    confiance = np.maximum(p, 1 - p)
    forte = confiance >= CONFIANCE_EXAMINEE
    fp = int(np.sum((y == 0) & (prediction == 1)))
    fn = int(np.sum((y == 1) & (prediction == 0)))
    tn, tp = int(np.sum(y == 0)) - fp, int(np.sum(y == 1)) - fn
    f1_normal = 2 * tn / (2 * tn + fp + fn)
    f1_pneumonie = 2 * tp / (2 * tp + fp + fn)
    mesures = {
        "n_images": int(len(y)),
        "brier": float(brier_image.mean()),
        "log_loss": float(perte_image.mean()),
        "ece_pneumonie_10_intervalles": float(ece),
        "score_pneumonie_moyen": float(p.mean()),
        "proportion_PNEUMONIA": float(y.mean()),
        "n_confiance_090": int(forte.sum()),
        "n_erreurs_confiance_090": int(np.sum(forte & erreur)),
        "n_FP_confiance_090": int(np.sum(forte & (y == 0) & (prediction == 1))),
        "n_FN_confiance_090": int(np.sum(forte & (y == 1) & (prediction == 0))),
        "fraction_confiance_090": float(forte.mean()),
        "taux_erreur_confiance_090": float(erreur[forte].mean()) if forte.any() else np.nan,
        "faux_positifs": fp,
        "faux_negatifs": fn,
        "f1_macro": float((f1_normal + f1_pneumonie) / 2),
        "rappel_pneumonie": float(tp / (tp + fn)),
        "specificite_NORMAL": float(tn / (tn + fp)),
    }
    for classe, etiquette in CLASSES.items():
        masque = y == etiquette
        mesures[f"brier_{classe}"] = float(brier_image[masque].mean())
        mesures[f"log_loss_{classe}"] = float(perte_image[masque].mean())
    return mesures, courbe

print("Calculs prêts : Brier, perte logarithmique, calibration et erreurs à score élevé.")

## 3. Analyser les 84 fichiers de prédictions

La cellule vérifie chaque fichier puis recalcule les mesures. Les matrices de confusion doivent retrouver celles de la comparaison précédente. Les douze modèles restent distincts : les graines ne constituent pas 12 jeux de nouvelles images.

In [ ]:
lignes, courbes, inventaire = [], [], []
for variante, dossier in SOURCES.items():
    for nom, graine, condition in itertools.product(MODELES, GRAINES, CONDITIONS):
        chemin = dossier / "predictions" / f"{nom}_graine_{graine}__{condition}.csv"
        exiger(
            chemin.is_file(),
            f"Prédictions introuvables : {chemin}\n"
            "Utiliser le notebook 08 (sans augmentation) ou 10 (avec augmentation) "
            "pour recréer les prédictions depuis les modèles sauvegardés.",
        )
        tableau = pd.read_csv(
            chemin, dtype={"groupe": str, "chemin_relatif": str},
            float_precision="round_trip",
        )
        verifier_predictions(tableau, variante, nom, graine, condition)
        mesures, courbe = calculer_fiabilite(tableau)
        reference = BILAN.loc[(nom, graine, condition)]
        for metrique in [
            "faux_positifs", "faux_negatifs", "f1_macro",
            "rappel_pneumonie", "specificite_NORMAL",
        ]:
            exiger(
                abs(mesures[metrique] - reference[f"{metrique}_{variante}"]) < 1e-10,
                f"La mesure {metrique} ne retrouve pas la comparaison : {chemin.name}.",
            )
        identification = {
            "modele": nom, "graine": graine, "condition": condition,
            "augmentation": variante,
            "epoque": int(INFORMATIONS[variante][(nom, graine)]["epoque"]),
        }
        lignes.append({**identification, **mesures})
        for cle, valeur in identification.items():
            courbe[cle] = valeur
        courbes.append(courbe)
        inventaire.append({
            **identification,
            "fichier_relatif": chemin.relative_to(PROJET).as_posix(),
            "sha256": empreinte_fichier(chemin),
        })

resultats = pd.DataFrame(lignes)
intervalles = pd.concat(courbes, ignore_index=True)
exiger(
    len(resultats) == 84
    and not resultats.duplicated(CLES + ["augmentation"]).any(),
    "Nombre d’analyses incorrect.",
)
PROTOCOLE = {
    "version": "fiabilite_scores_v1",
    "ensemble": "val", "n_images": len(val_df), "classes": CLASSES,
    "sha256_val_csv": SHA_VAL,
    "sha256_comparaison_robustesse": empreinte_fichier(COMPARAISON),
    "protocoles_predictions": IDS,
    "predictions": inventaire,
    "graines": GRAINES,
    "intervalles_calibration": BORNES.tolist(),
    "convention_intervalles": "borne gauche incluse, droite exclue; dernier intervalle inclut 1",
    "brier": "moyenne par image de (score_PNEUMONIA - etiquette)**2; echelle binaire [0,1]",
    "log_loss": "moyenne par image, sans poids de classe; logaddexp sur marge signee",
    "ece": "PNEUMONIA; somme n_intervalle/N * abs(frequence - score_moyen)",
    "confiance_examinee": CONFIANCE_EXAMINEE,
    "seuil_prediction": "argmax des logits; NORMAL en cas d’egalite",
    "versions": {"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__},
    "limite": (
        "Diagnostic descriptif sur validation reutilisee pour selection et choix de "
        "l’augmentation; test deja consulte. Aucun ajustement de calibration. "
        "Moyennes et ecarts-types entre graines, pas intervalles de confiance patients."
    ),
}
SORTIE.mkdir(parents=True, exist_ok=True)
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(
        json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
        "Les sources ou la méthode ont changé. Conserver ce dossier et choisir "
        "un nouveau nom SORTIE pour cette nouvelle analyse.",
    )
else:
    ecrire_json(PROTOCOLE, fichier_protocole)
ecrire_csv(resultats, SORTIE / "resultats_fiabilite.csv")
ecrire_csv(intervalles, SORTIE / "intervalles_calibration.csv")
print("84 fichiers vérifiés et analysés.")

## 4. Comparer les deux variantes d’apprentissage

Les différences sont calculées **avec augmentation − sans augmentation**, pour la même architecture, la même graine et la même condition.

Pour le Brier, la perte logarithmique et l’écart de calibration, une différence **négative** est favorable. La baisse du nombre d’erreurs à score élevé doit se lire avec la proportion d’images qui reçoivent un score élevé.

La moyenne et l’écart-type décrivent les trois graines. Un second tableau reprend uniquement 123 et 2026, puisque la graine 42 avait déjà servi au choix des hyperparamètres. Cette séparation ne rend pas ces observations indépendantes de la sélection sur validation.

In [ ]:
MESURES = [
    "brier", "log_loss", "ece_pneumonie_10_intervalles",
    "brier_NORMAL", "brier_PNEUMONIA", "log_loss_NORMAL", "log_loss_PNEUMONIA",
    "n_confiance_090", "n_erreurs_confiance_090",
    "n_FP_confiance_090", "n_FN_confiance_090",
    "fraction_confiance_090", "taux_erreur_confiance_090",
    "f1_macro", "rappel_pneumonie", "specificite_NORMAL",
]

def comparer_resultats(tableau):
    colonnes = CLES + ["epoque", "n_images"] + MESURES
    sans = tableau[tableau["augmentation"] == "sans"][colonnes]
    avec = tableau[tableau["augmentation"] == "avec"][colonnes]
    comparaison = sans.merge(
        avec, on=CLES, suffixes=("_sans", "_avec"), validate="one_to_one"
    )
    exiger(
        len(sans) == len(avec) == len(comparaison) == 42,
        "Comparaison incomplète entre les deux variantes.",
    )
    for metrique in MESURES:
        comparaison[f"difference_{metrique}"] = (
            comparaison[f"{metrique}_avec"] - comparaison[f"{metrique}_sans"]
        )
    return comparaison

def synthetiser(comparaison):
    colonnes = [
        f"{m}_{suffixe}" for m in MESURES for suffixe in ["sans", "avec"]
    ] + [f"difference_{m}" for m in MESURES]
    # Les comptes permettent de repérer une mesure non définie pour une graine.
    synthese = comparaison.groupby(["modele", "condition"])[colonnes].agg(
        ["mean", "std", "count"]
    )
    synthese.columns = [f"{nom}_{stat}" for nom, stat in synthese.columns]
    return synthese.reset_index()

comparaison = comparer_resultats(resultats)
synthese = synthetiser(comparaison)
synthese_nouvelles = synthetiser(comparaison[comparaison["graine"] != 42])
ecrire_csv(comparaison, SORTIE / "comparaison_fiabilite.csv")
ecrire_csv(synthese, SORTIE / "synthese_fiabilite.csv")
ecrire_csv(synthese_nouvelles, SORTIE / "synthese_fiabilite_graines_123_2026.csv")

colonnes_affichees = ["modele", "condition"] + [
    f"{m}_{suffixe}_mean"
    for m in ["brier", "log_loss", "ece_pneumonie_10_intervalles"]
    for suffixe in ["sans", "avec"]
]
display(
    synthese[synthese["condition"].isin(["originale", "luminosite_090"])]
    [colonnes_affichees].round(5)
)

## 5. Afficher les courbes et leurs effectifs

Deux figures sont produites : images originales et luminosité × 0,90. Chaque panneau correspond à **un modèle et une graine**, avec les variantes sans et avec augmentation.

- La diagonale représente l’accord entre score et fréquence observée.
- Un point sous la diagonale indique un score PNEUMONIA supérieur à la fréquence observée dans cet intervalle ; au-dessus, il est inférieur.
- L’histogramme sous chaque courbe montre le nombre d’images par intervalle. Une courbe irrégulière dans des intervalles peu remplis appelle une lecture prudente.

Les valeurs détaillées restent disponibles pour les sept conditions dans **intervalles_calibration.csv**.

In [ ]:
def tracer_calibration(condition):
    couleurs = {"sans": "#2563EB", "avec": "#D97706"}
    noms = {"CNN_classique": "CNN classique", "CNN_CBAM": "CNN avec CBAM"}
    fig, axes = plt.subplots(
        4, 3, figsize=(15, 10.5),
        gridspec_kw={"height_ratios": [3, 1, 3, 1]},
    )
    fig.subplots_adjust(top=0.865, bottom=0.06, hspace=0.80, wspace=0.32)
    for i, nom in enumerate(MODELES):
        for j, graine in enumerate(GRAINES):
            axe, histogramme = axes[2 * i, j], axes[2 * i + 1, j]
            axe.plot([0, 1], [0, 1], "--", color="#6B7280", linewidth=1)
            for variante, decalage in [("sans", 0.03), ("avec", 0.07)]:
                selection = intervalles[
                    (intervalles["modele"] == nom)
                    & (intervalles["graine"] == graine)
                    & (intervalles["condition"] == condition)
                    & (intervalles["augmentation"] == variante)
                ].sort_values("intervalle")
                axe.plot(
                    selection["score_moyen"], selection["fraction_PNEUMONIA"],
                    "o-", color=couleurs[variante], linewidth=1.4, markersize=4,
                )
                histogramme.bar(
                    selection["borne_inferieure"] + decalage,
                    selection["n_images"], width=0.035,
                    color=couleurs[variante], alpha=0.8,
                )
            axe.set(
                title=f"{noms[nom]} · graine {graine}",
                xlim=(-0.02, 1.02), ylim=(-0.02, 1.02),
                ylabel="Fréquence PNEUMONIA",
                xlabel="Score PNEUMONIA moyen",
            )
            histogramme.set(
                xlim=(-0.02, 1.02), ylabel="Images",
                xlabel="Intervalle de score PNEUMONIA",
            )
            axe.grid(alpha=0.2)
            histogramme.grid(axis="y", alpha=0.2)
    legendes = [
        Line2D([0], [0], color=couleurs["sans"], marker="o", label="Sans augmentation"),
        Line2D([0], [0], color=couleurs["avec"], marker="o", label="Avec augmentation"),
    ]
    fig.legend(
        handles=legendes, loc="upper center", bbox_to_anchor=(0.5, 0.925),
        ncol=2, frameon=False,
    )
    fig.suptitle(
        f"Calibration sur validation · {LIBELLES[condition]}\n"
        "10 intervalles fixes et effectifs par intervalle",
        fontsize=14, y=0.98,
    )
    destination = SORTIE / f"calibration_{condition}.png"
    fig.savefig(destination, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return destination

for condition in ["originale", "luminosite_090"]:
    print("Figure :", tracer_calibration(condition))

ecrire_json(
    {"analyse": "fiabilite_scores_v1", "n_analyses": len(resultats),
     "n_comparaisons": len(comparaison), "id_protocole": id_protocole(PROTOCOLE)},
    SORTIE / "termine.json",
)
print("Analyse terminée.")
print("Fichier à transmettre :", SORTIE / "comparaison_fiabilite.csv")

## Ce que cette étape permettra de décider

Nous chercherons si une amélioration du F1 s’accompagne de scores plus fiables, et si les images assombries augmentent les erreurs à score élevé. Les tableaux distinguent les classes, les perturbations et les graines.

Un score proche de 0,90 n’est pas automatiquement une probabilité clinique de 90 %. La calibration doit être évaluée dans la population et les conditions d’utilisation concernées. Ici, l’analyse porte sur les étiquettes et la répartition du dataset.

Si un défaut de calibration apparaît, un ajustement des scores pourra être étudié avec une séparation par groupes entre les observations utilisées pour l’ajustement et celles utilisées pour son évaluation. La sélection antérieure des modèles sur cette validation devra rester explicitement signalée. Ce notebook ne crée pas cette séparation et ne présente pas de résultat de calibration ajustée.

**À transmettre :** **comparaison_fiabilite.csv**, puis, si possible, **calibration_originale.png** et **calibration_luminosite_090.png**. Le fichier **intervalles_calibration.csv** contient les effectifs et les coordonnées de toutes les courbes.